In [1]:
import pandas as pd
import xarray as xr
import xcdat as xc
import numpy as np
import os
import glob
import re
import cftime

/global/homes/j/jungchoi/.conda/envs/pcmdi_metrics/lib/python3.10/site-packages/esmpy/interface/loadESMF.py:94: VersionWarning: ESMF installation version 8.8.0, ESMPy version 8.8.0b0
  warnings.warn("ESMF installation version {}, ESMPy version {}".format(


Model information, Target lead time

In [2]:
print("%%%%%%%% start %%%%%%%")
output_dir = "/pscratch/sd/j/jungchoi/DCPP/_metrics"
output_grid = xc.regridder.grid.create_uniform_grid(-88.75, 88.75, 2.5, 0.0, 357.5, 2.5)
output_grid_no = "144x72"

input_dir = "/pscratch/sd/j/jungchoi/OBS"

initialization_year_start = 1960
initialization_year_end = 2016


%%%%%%%% start %%%%%%%


# Target leadtime average, Horizonal interpolarion for each ensemble members and initializations

In [14]:
data_name = "ERA5"
#obs_var_name = "t2m"
#out_var_name = "tas"
#obs_var_name = "MSL"
#out_var_name = "psl"

data_name = "GPCP"
obs_var_name = "precip"
out_var_name = "pr"
initialization_year_start = 1978

lead_year_end = 5

target_year_start = initialization_year_start + 1
target_year_end   = initialization_year_end + lead_year_end 

target_years = list(range(target_year_start, target_year_end + 1))

each_ds = []
        
for year in target_years:   
    if obs_var_name == "t2m" or obs_var_name == "precip":
        file = os.path.join(input_dir, data_name, f"{obs_var_name}.{year}.{output_grid_no}.nc")
        ds = xr.open_dataset(file)
           
    if obs_var_name == "MSL":
        file_list = os.path.join(input_dir, data_name, f"{obs_var_name}.{year}*.nc")
        ds = xr.open_mfdataset(file_list,combine='by_coords')
        #print(ds) 
     
    each_ds.append(ds)    
    ds.close()

all_ds = xr.concat(each_ds, dim="time")
print(all_ds)
print(all_ds.time)

annual_ds = all_ds.coarsen(time=12, boundary='trim').mean()
years = annual_ds.time.dt.year.values
new_time = np.array([np.datetime64(f"{y}-01-01") for y in years])
annual_ds = annual_ds.assign_coords(time=new_time)
annual_ds = annual_ds.rename({obs_var_name: out_var_name})


if out_var_name == "psl":
    annual_ds = annual_ds.regridder.horizontal(f"{out_var_name}", output_grid, tool="regrid2")
    annual_ds = annual_ds * 0.01    
print(annual_ds)
    
    
# Save to NetCDF for each lead time
output_filename = f"{output_dir}/OBS/{out_var_name}.{output_grid_no}.{target_year_start}-{target_year_end}.ann.nc"
if os.path.exists(output_filename):
    os.remove(output_filename)
annual_ds.to_netcdf(output_filename)
print(f"%% OBS anmnual mean dataset saved: {output_filename}")

<xarray.Dataset> Size: 23MB
Dimensions:   (time: 516, lat: 72, lon: 144, bnds: 2)
Coordinates:
  * time      (time) datetime64[ns] 4kB 1979-01-01 1979-02-01 ... 2021-12-01
  * lat       (lat) float64 576B -88.75 -86.25 -83.75 ... 83.75 86.25 88.75
  * lon       (lon) float64 1kB 0.0 2.5 5.0 7.5 10.0 ... 350.0 352.5 355.0 357.5
Dimensions without coordinates: bnds
Data variables:
    precip    (time, lat, lon) float32 21MB 0.0 0.0 0.0 ... 0.2785 0.2782 0.2785
    lon_bnds  (time, lon, bnds) float64 1MB -1.25 1.25 1.25 ... 356.2 358.8
    lat_bnds  (time, lat, bnds) float64 594kB -90.0 -87.5 -87.5 ... 87.5 90.0
Attributes: (12/18)
    Conventions:           CF-1.0
    curator:               Dr. Jian-Jian Wang\nESSIC, University of Maryland ...
    citation:              Adler, R.F., G.J. Huffman, A. Chang, R. Ferraro, P...
    title:                 GPCP Version 2.3 Combined Precipitation Dataset (F...
    platform:              NOAA POES (Polar Orbiting Environmental Satellites)
    sou